# Feature Engineering and Modelling

---

1. Import packages
2. Load data
3. Modelling

---

## 1. Import packages

In [4]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt

# Shows plots in jupyter notebook
%matplotlib inline

# Set plot style
sns.set(color_codes=True)

---
## 2. Load data

### Data sampling

The first thing we want to do is split our dataset into training and test samples. The reason why we do this, is so that we can simulate a real life situation by generating predictions for our test sample, without showing the predictive model these data points. This gives us the ability to see how well our model is able to generalise to new data, which is critical.

A typical % to dedicate to testing is between 20-30, for this example we will use a 75-25% split between train and test respectively.

In [5]:
# Make a copy of our data
train_df = df.copy()

# Separate target variable from independent variables
y = df['churn']
X = df.drop(columns=['id', 'churn'])
print(X.shape)
print(y.shape)

(14606, 61)
(14606,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(10954, 61)
(10954,)
(3652, 61)
(3652,)


### Model training

### Evaluation & Model Comparison

Now let's evaluate how well each trained model performs on the test dataset and identify the best model.

In [ ]:
# Import required for modeling
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn import metrics
import xgboost as xgb
import time

# Fix imports in the notebook
print("✓ All modeling packages imported successfully")

✓ All modeling packages imported successfully


In [ ]:
print("=" * 90)
print("MODEL TRAINING WITH HYPERPARAMETER TUNING")
print("=" * 90)

# Dictionary to store all models
models_dict = {}

# ============================================================================
# MODEL 1: RANDOM FOREST CLASSIFIER WITH GRIDSEARCHCV
# ============================================================================
print("\n[1/3] Random Forest Classifier with GridSearchCV...")
start_time = time.time()

rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)
rf_best = rf_grid.best_estimator_
rf_time = time.time() - start_time

print(f"\n✓ Random Forest trained in {rf_time:.2f}s")
print(f"  Best CV Score (ROC-AUC): {rf_grid.best_score_:.4f}")
print(f"  Best Parameters: {rf_grid.best_params_}")

# Evaluate RF
rf_pred_prob = rf_best.predict_proba(X_test)[:, 1]
rf_pred = rf_best.predict(X_test)
rf_metrics = {
    'model': rf_best,
    'params': rf_grid.best_params_,
    'cv_score': rf_grid.best_score_,
    'roc_auc': metrics.roc_auc_score(y_test, rf_pred_prob),
    'accuracy': metrics.accuracy_score(y_test, rf_pred),
    'precision': metrics.precision_score(y_test, rf_pred),
    'recall': metrics.recall_score(y_test, rf_pred),
    'f1': metrics.f1_score(y_test, rf_pred),
    'time': rf_time
}

models_dict['Random Forest'] = rf_metrics

print(f"\n  Test ROC-AUC:   {rf_metrics['roc_auc']:.4f}")
print(f"  Test Accuracy:  {rf_metrics['accuracy']:.4f}")

# ============================================================================
# MODEL 2: XGBOOST CLASSIFIER WITH GRIDSEARCHCV
# ============================================================================
print("\n[2/3] XGBoost Classifier with GridSearchCV...")
start_time = time.time()

xgb_params = {
    'max_depth': [5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5]
}

xgb_grid = GridSearchCV(
    xgb.XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss'),
    xgb_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)
xgb_best = xgb_grid.best_estimator_
xgb_time = time.time() - start_time

print(f"\n✓ XGBoost trained in {xgb_time:.2f}s")
print(f"  Best CV Score (ROC-AUC): {xgb_grid.best_score_:.4f}")
print(f"  Best Parameters: {xgb_grid.best_params_}")

# Evaluate XGB
xgb_pred_prob = xgb_best.predict_proba(X_test)[:, 1]
xgb_pred = xgb_best.predict(X_test)
xgb_metrics = {
    'model': xgb_best,
    'params': xgb_grid.best_params_,
    'cv_score': xgb_grid.best_score_,
    'roc_auc': metrics.roc_auc_score(y_test, xgb_pred_prob),
    'accuracy': metrics.accuracy_score(y_test, xgb_pred),
    'precision': metrics.precision_score(y_test, xgb_pred),
    'recall': metrics.recall_score(y_test, xgb_pred),
    'f1': metrics.f1_score(y_test, xgb_pred),
    'time': xgb_time
}

models_dict['XGBoost'] = xgb_metrics

print(f"\n  Test ROC-AUC:   {xgb_metrics['roc_auc']:.4f}")
print(f"  Test Accuracy:  {xgb_metrics['accuracy']:.4f}")

# ============================================================================
# MODEL 3: GRADIENT BOOSTING CLASSIFIER WITH GRIDSEARCHCV
# ============================================================================
print("\n[3/3] Gradient Boosting Classifier with GridSearchCV...")
start_time = time.time()

gb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'subsample': [0.7, 0.8, 0.9],
    'max_features': ['sqrt', 'log2']
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

gb_grid.fit(X_train, y_train)
gb_best = gb_grid.best_estimator_
gb_time = time.time() - start_time

print(f"\n✓ Gradient Boosting trained in {gb_time:.2f}s")
print(f"  Best CV Score (ROC-AUC): {gb_grid.best_score_:.4f}")
print(f"  Best Parameters: {gb_grid.best_params_}")

# Evaluate GB
gb_pred_prob = gb_best.predict_proba(X_test)[:, 1]
gb_pred = gb_best.predict(X_test)
gb_metrics = {
    'model': gb_best,
    'params': gb_grid.best_params_,
    'cv_score': gb_grid.best_score_,
    'roc_auc': metrics.roc_auc_score(y_test, gb_pred_prob),
    'accuracy': metrics.accuracy_score(y_test, gb_pred),
    'precision': metrics.precision_score(y_test, gb_pred),
    'recall': metrics.recall_score(y_test, gb_pred),
    'f1': metrics.f1_score(y_test, gb_pred),
    'time': gb_time
}

models_dict['Gradient Boosting'] = gb_metrics

print(f"\n  Test ROC-AUC:   {gb_metrics['roc_auc']:.4f}")
print(f"  Test Accuracy:  {gb_metrics['accuracy']:.4f}")

print("\n" + "=" * 90)
print("✓ All models trained successfully!")
print("=" * 90)

MODEL TRAINING WITH HYPERPARAMETER TUNING

[1/3] Random Forest Classifier with GridSearchCV...
Fitting 5 folds for each of 216 candidates, totalling 1080 fits


/home/josh/Forage/PowerCo Churn Prediction /venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/josh/Forage/PowerCo Churn Prediction /venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/josh/Forage/PowerCo Churn Prediction /venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.w


✓ Random Forest trained in 1350.39s
  Best CV Score (ROC-AUC): 0.7006
  Best Parameters: {'max_depth': 15, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}

  Test ROC-AUC:   0.6678
  Test Accuracy:  0.9022

[2/3] XGBoost Classifier with GridSearchCV...
Fitting 5 folds for each of 729 candidates, totalling 3645 fits

✓ XGBoost trained in 2446.68s
  Best CV Score (ROC-AUC): 0.7133
  Best Parameters: {'colsample_bytree': 0.9, 'learning_rate': 0.01, 'max_depth': 9, 'min_child_weight': 1, 'n_estimators': 300, 'subsample': 0.8}

  Test ROC-AUC:   0.6773
  Test Accuracy:  0.9055

[3/3] Gradient Boosting Classifier with GridSearchCV...
Fitting 5 folds for each of 1458 candidates, totalling 7290 fits

✓ Gradient Boosting trained in 6349.46s
  Best CV Score (ROC-AUC): 0.7081
  Best Parameters: {'learning_rate': 0.01, 'max_depth': 7, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 300, 'subsample': 0.9}

  Test R

In [ ]:
print("\n" + "=" * 90)
print("MODEL COMPARISON & BEST MODEL SELECTION")
print("=" * 90)

# Create comparison dataframe
comparison_data = []
for model_name, metrics_dict in models_dict.items():
    comparison_data.append({
        'Model': model_name,
        'CV ROC-AUC': metrics_dict['cv_score'],
        'Test ROC-AUC': metrics_dict['roc_auc'],
        'Accuracy': metrics_dict['accuracy'],
        'Precision': metrics_dict['precision'],
        'Recall': metrics_dict['recall'],
        'F1-Score': metrics_dict['f1'],
        'Training Time (s)': metrics_dict['time']
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "MODEL PERFORMANCE SUMMARY:")
print("-" * 90)
print(comparison_df.to_string(index=False))

# Find best model by test ROC-AUC
best_idx = comparison_df['Test ROC-AUC'].idxmax()
best_model_name = comparison_df.loc[best_idx, 'Model']
best_model = models_dict[best_model_name]['model']
best_params = models_dict[best_model_name]['params']
best_roc_auc = models_dict[best_model_name]['roc_auc']

print("\n" + "=" * 90)
print(f"🏆 BEST MODEL: {best_model_name}")
print("=" * 90)
print(f"\nTest ROC-AUC Score: {best_roc_auc:.4f}")
print(f"\nPerformance Metrics:")
print(f"  • Accuracy:  {models_dict[best_model_name]['accuracy']:.4f}")
print(f"  • Precision: {models_dict[best_model_name]['precision']:.4f}")
print(f"  • Recall:    {models_dict[best_model_name]['recall']:.4f}")
print(f"  • F1-Score:  {models_dict[best_model_name]['f1']:.4f}")

print(f"\nBest Hyperparameters:")
for param, value in best_params.items():
    print(f"  • {param}: {value}")

# Feature importance
if hasattr(best_model, 'feature_importances_'):
    print("\n" + "=" * 90)
    print("TOP 15 MOST IMPORTANT FEATURES:")
    print("=" * 90)
    
    feature_importance_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\n")
    for idx, (_, row) in enumerate(feature_importance_df.head(15).iterrows(), 1):
        print(f"{idx:2d}. {row['Feature']:40s} {row['Importance']:.6f}")
    
    # Save feature importance
    feature_importance_df.to_csv('./feature_importance.csv', index=False)
    print("\n✓ Feature importance saved to: feature_importance.csv")

# Save comparison results
comparison_df.to_csv('./model_comparison.csv', index=False)
print("\n✓ Model comparison saved to: model_comparison.csv")

print("\n" + "=" * 90)
print("✓ Modeling Complete!")
print("=" * 90)


MODEL COMPARISON & BEST MODEL SELECTION

MODEL PERFORMANCE SUMMARY:
------------------------------------------------------------------------------------------
            Model  CV ROC-AUC  Test ROC-AUC  Accuracy  Precision   Recall  F1-Score  Training Time (s)
    Random Forest    0.700628      0.667778  0.902245   0.909091 0.027322  0.053050        1350.388762
          XGBoost    0.713266      0.677289  0.905531   0.956522 0.060109  0.113111        2446.681137
Gradient Boosting    0.708131      0.676010  0.901698   1.000000 0.019126  0.037534        6349.459585

🏆 BEST MODEL: XGBoost

Test ROC-AUC Score: 0.6773

Performance Metrics:
  • Accuracy:  0.9055
  • Precision: 0.9565
  • Recall:    0.0601
  • F1-Score:  0.1131

Best Hyperparameters:
  • colsample_bytree: 0.9
  • learning_rate: 0.01
  • max_depth: 9
  • min_child_weight: 1
  • n_estimators: 300
  • subsample: 0.8

TOP 15 MOST IMPORTANT FEATURES:


 1. margin_net_pow_ele                       0.051034
 2. origin_up_kamkkxfxx